# Source Apportionment Modeling with State Space Models

#### *By George Whittington*

---

## What is a State Space Model?

In standard regression, we assume the data is what it is measured. There is no uncertainty in that. For example, if we have a measurement from a sensor that says $PM_{2.5}$ concentration at time $t$ is $15.6 \mu g / m^3$, then that is how much is in the air. But state space models allow uncertainty.

A State Space Model admits the real world is messy and our sensor data is imperfect. It splits reality into two parts:

1. **The Hidden State (The Truth)**: This is the actually, physical amount of pollution floating in the air in say Baltimore or St. Louis. The thing here is we can never see this amount perfectly

2. **The Observation (The Sensor)**: This is he number the actually sensor spits out. It is trying to measure the truth, but it gets off by things like sensor drift, calibration errors, or random electronic noise

The goal of this model is to look at the noisy observations and mathematically guess the hidden truth

## How it Works: Kalman Filtering

To guess the hidden truth, the model uses an algorithm called a Kalman Filter. This essentially a continuous and iterative process with two parts:

1. **The Prediction (Blind Guess)**: Before looking at the sensor data for today, the math looks back at yesterday, and uses that to guess what it should be today

2. **The Update (Reality Check)**: The model now looks at the sensor reading for today and then checks the uncertainty for how reliable that reading is:

    - If the sensor is highly accurate (low uncertainty), the model throws away its blind guess and believes the sensor

    - If the answer is highly inaccurate (high uncertainty), the model ignores the sensor and sicks closer to its mathematical guess

It would then repeat this predict-update cycle for every single time stamp in the data set

## Modeling Process (`statsmodels`)

Beyond the theory, to translate the "hidden truth vs messy sensor" in to a mathematical sense that can be implemented, the `statsmodels` library relies o a generalized system of two core equations. Together, they can describe how the real world changes an how our sensors interact with it.

1. **The State Equation (How reality changes)**

    This equation describes how the physical, hidden truth naturally evolves from the current moment ($t$) to the next moment ($t + 1$)

    $$
        \alpha_{t + 1} = T_t \alpha_t + c_t + R_t \eta_t
    $$

    - $\alpha_t$: The **Hidden State** at the current time (e.g., the true pollution level)

    - $T_t$: The transition matrix. This represents the natural physics of the system (e.g., how much of today's smog normally lingers tomorrow)

    - $c_t$: The state intercept, representing a constant base line shift

    - $R_t \eta_t$: The **Process Noise** (where $\eta_t$ is the noise and $R_t$ is a selection matrix). This represents the unpredictable randomness of reality, like a sudden gust of wind. The model assumes this noise follows a normal distribution with a covariance of $Q_t$

2. **The Observation Equation (How the Sensor Reads Reality)**

    This equation describes what the sensor actually spits out when it tries to look at the hidden state

    $$
        y_t = Z_t \alpha_t + d_t + \epsilon_t
    $$

    - $y_t$: The **Observation** at the current time (the number of the sensor's screen)

    - $Z_t$: The design matrix. This is how the sensor translates reality into data (e.g., converting physical particles into electrical voltage)

    - $d_t$: The observation intercept (e.g., the sensor's baseline calibration)

    - $\epsilon_t$: The **Observation Noise**. This is the sensor's error. Things like calibration drift or electronic static. The model assumes the error follows a normal distribution with a covariance of $H_t$

**Inputs vs. Outputs**

When we actually run the model in code, we have to separate what we know from what we want to find out:

- **The Input**: We feed the model two distinct pieces of the real world:

    1. The messy sensor data readings ($y_t$)

    2. The uncertainty of that given measurement, which tells the model exactly how unreliable those sensor readings are at any given time. In the math, this is our observation noise covariance, plugging directly into $H_t$

- **The Output**: Because the model has both measurements and uncertainty, the algorithm does not have to guess at the observation noise ($\epsilon_t$). It can dedicate the computation to the model's best guess that the hidden truth ($\alpha_t$) and its process noise ($\eta_t$)